In [ ]:
import tensorflow as tf

# Load + preprocess data
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[..., tf.newaxis] / 255.0
x_test = x_test[..., tf.newaxis] / 255.0

# Define model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10)
])

# Compile model
model.compile(optimizer='adam',
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=['accuracy'])

# Train
model.fit(x_train, y_train, epochs=5, validation_split=0.1)

# Evaluate
model.evaluate(x_test, y_test)


: 

In [ ]:
model.save("saved_model/mnist_model")


In [ ]:
import tensorflow as tf
import numpy as np

# Load saved model (SavedModel format)
model_loaded = tf.saved_model.load("tensorflow_model")

# Get the inference function
infer = model_loaded.signatures["serving_default"]

# Check output key
img = tf.constant(x_test[0:1].reshape(1, 1, 28, 28), dtype=tf.float32)
out = infer(input=img)
print("Output keys:", out.keys())  # Important!

# Use correct key here
output_key = list(out.keys())[0]

# Predict (example for first 100 images)
predictions = []
for i in range(len(x_test)):
    img = tf.convert_to_tensor(x_test[i:i+1].reshape(1, 1, 28, 28), dtype=tf.float32)
    output = infer(input = img)
    pred = tf.argmax(output[output_key], axis=1).numpy()[0]
    predictions.append(pred)

# Evaluate manually
true_labels = y_test[:len(predictions)]
accuracy = np.mean(np.array(predictions) == true_labels)
print("Accuracy:", accuracy*100)
